# CYP3A4 Site-of-Metabolism Predictor

## Quantum Liquid Neural Network with Reverse Geometry Learning

This notebook trains a state-of-the-art model for predicting sites of metabolism in CYP3A4 substrates.

### Key Innovations:
1. **Reverse Geometry Learning**: Infer enzyme pocket dynamics from known SoM data
2. **Liquid Neural Network**: Continuous-time ODE dynamics for molecular encoding
3. **Graph Laplacian Features**: Topological peripherality and flexibility
4. **Multi-task Training**: Ranking + contrastive + geometry losses

### Requirements:
- Google Colab Pro+ with A100 or RTX 6000
- ~2 hours training time for 100 epochs

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q rdkit-pypi
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

In [ ]:
# Clone repository (replace with your repo URL)
!git clone https://github.com/Nikku03/enzyme_Software.git
%cd enzyme_Software

In [ ]:
# Verify setup
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download Data (if needed)

In [ ]:
import os

# Check if data exists
data_path = 'data/curated/merged_cyp3a4_extended.json'
enzyme_path = '1W0E.pdb'

if not os.path.exists(data_path):
    print("Data not found! Please upload merged_cyp3a4_extended.json")
else:
    print(f"✓ Data found: {data_path}")

if not os.path.exists(enzyme_path):
    # Download enzyme structure
    !wget -q https://files.rcsb.org/download/1W0E.pdb
    print("Downloaded 1W0E.pdb")
else:
    print(f"✓ Enzyme structure found: {enzyme_path}")

## 3. Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Model
    'hidden_dim': 128,
    'n_lnn_layers': 3,
    'n_gnn_layers': 4,
    'n_attention_heads': 4,
    'dropout': 0.1,
    
    # Training
    'batch_size': 32,  # Increase for RTX 6000
    'epochs': 100,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    
    # Loss weights
    'ranking_weight': 1.0,
    'contrastive_weight': 0.5,
    'reverse_geometry_weight': 0.3,
    
    # Hardware
    'device': 'cuda',
    'mixed_precision': True,
    'num_workers': 4,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4. Train Model

In [ ]:
# Run training
!python colab_train_cyp3a4.py \
    --epochs {CONFIG['epochs']} \
    --batch_size {CONFIG['batch_size']} \
    --hidden_dim {CONFIG['hidden_dim']} \
    --n_lnn_layers {CONFIG['n_lnn_layers']} \
    --n_gnn_layers {CONFIG['n_gnn_layers']} \
    --lr {CONFIG['lr']} \
    --dropout {CONFIG['dropout']}

## 5. Evaluate Best Model

In [ ]:
import torch
import json
from colab_train_cyp3a4 import (
    Config, EnzymePocket, CYP3A4Dataset, CYP3A4SoMPredictor,
    collate_fn, compute_accuracy
)
from torch.utils.data import DataLoader

# Load best model
checkpoint = torch.load('checkpoints/best_model.pt')
print(f"Best model from epoch {checkpoint['epoch']}")
print(f"Validation metrics: {checkpoint['val_metrics']}")

# Create model
config = Config(**checkpoint['config'])
pocket = EnzymePocket(config.enzyme_pdb)
model = CYP3A4SoMPredictor(config, pocket)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(config.device)
model.eval()

print("\n✓ Model loaded successfully!")

In [ ]:
# Evaluate on all datasets
results = {}

for source in ['AZ120', 'Zaretzki', 'DrugBank', 'MetXBioDB']:
    dataset = CYP3A4Dataset(config.data_path, pocket, sources=[source])
    if len(dataset) == 0:
        continue
    
    loader = DataLoader(
        dataset, batch_size=32, shuffle=False,
        collate_fn=collate_fn, num_workers=0
    )
    
    total_top1 = 0
    total_top3 = 0
    n_batches = 0
    
    with torch.no_grad():
        for batch in loader:
            x = batch['x'].to(config.device)
            coords = batch['coords'].to(config.device)
            accessibility = batch['accessibility'].to(config.device)
            valid_mask = batch['valid_mask'].to(config.device)
            som_labels = batch['som_labels'].to(config.device)
            edge_index = batch['edge_index'].to(config.device)
            edge_attr = batch['edge_attr'].to(config.device)
            batch_ptr = batch['batch_ptr'].to(config.device)
            
            scores, _, _ = model(
                x, coords, accessibility, valid_mask,
                edge_index, edge_attr, batch_ptr
            )
            
            top1, top3, _ = compute_accuracy(scores, som_labels, valid_mask)
            total_top1 += top1
            total_top3 += top3
            n_batches += 1
    
    results[source] = {
        'top1': total_top1 / n_batches * 100,
        'top3': total_top3 / n_batches * 100,
        'n_molecules': len(dataset),
    }

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"\n{'Dataset':<15} {'N':>8} {'Top-1':>10} {'Top-3':>10}")
print("-"*50)
for source, metrics in results.items():
    print(f"{source:<15} {metrics['n_molecules']:>8} {metrics['top1']:>9.1f}% {metrics['top3']:>9.1f}%")

## 6. Make Predictions on New Molecules

In [ ]:
from rdkit import Chem
from colab_train_cyp3a4 import mol_to_graph_data

def predict_som(smiles: str, model, pocket, device='cuda'):
    """Predict site of metabolism for a molecule."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Convert to graph
    graph_data = mol_to_graph_data(mol, pocket)
    
    # Batch for model
    batch = {
        'x': graph_data['x'].unsqueeze(0).to(device),
        'coords': graph_data['coords'].unsqueeze(0).to(device),
        'accessibility': graph_data['accessibility'].unsqueeze(0).to(device),
        'valid_mask': graph_data['valid_mask'].unsqueeze(0).to(device),
        'edge_index': graph_data['edge_index'].to(device),
        'edge_attr': graph_data['edge_attr'].to(device),
        'batch_ptr': torch.tensor([0, graph_data['n_atoms']]).to(device),
    }
    
    # Predict
    with torch.no_grad():
        scores, pose_scores, chem_scores = model(
            batch['x'], batch['coords'], batch['accessibility'],
            batch['valid_mask'], batch['edge_index'], batch['edge_attr'],
            batch['batch_ptr']
        )
    
    scores = scores[0].cpu().numpy()
    valid_mask = batch['valid_mask'][0].cpu().numpy()
    
    # Get ranking
    results = []
    for i in range(len(scores)):
        if valid_mask[i]:
            atom = mol.GetAtomWithIdx(i)
            results.append({
                'atom_idx': i,
                'symbol': atom.GetSymbol(),
                'score': scores[i],
            })
    
    results = sorted(results, key=lambda x: -x['score'])
    return results


# Example predictions
test_smiles = [
    ('Midazolam', 'Cn1cc(C(=O)c2ccccc2Cl)c3ccc(Cl)cc13'),
    ('Testosterone', 'CC12CCC3C(CCC4=CC(=O)CCC34C)C1CCC2O'),
    ('Caffeine', 'Cn1c(=O)c2c(ncn2C)n(C)c1=O'),
]

print("\nSoM Predictions:")
print("="*60)

for name, smiles in test_smiles:
    print(f"\n{name} ({smiles[:30]}...)")
    results = predict_som(smiles, model, pocket)
    if results:
        print(f"  Top 3 predicted sites:")
        for r in results[:3]:
            print(f"    Atom {r['atom_idx']} ({r['symbol']}): score={r['score']:.3f}")

## 7. Save Model for Deployment

In [ ]:
# Save complete model package
import pickle

package = {
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'pocket_data': {
        'fe_pos': pocket.fe_pos.tolist(),
        'feo_axis': pocket.feo_axis.tolist(),
        'shell_hydro': pocket.shell_hydro,
    },
    'results': results,
}

torch.save(package, 'cyp3a4_som_predictor.pt')
print("✓ Saved model to cyp3a4_som_predictor.pt")

# Download from Colab
from google.colab import files
files.download('cyp3a4_som_predictor.pt')

## 8. Analysis: What Did the Model Learn?

Let's visualize what the reverse geometry module learned about the enzyme pocket.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Analyze reverse geometry scores
all_pose_scores = []
all_som_labels = []

with torch.no_grad():
    for batch in loader:
        x = batch['x'].to(config.device)
        coords = batch['coords'].to(config.device)
        accessibility = batch['accessibility'].to(config.device)
        valid_mask = batch['valid_mask'].to(config.device)
        som_labels = batch['som_labels']
        edge_index = batch['edge_index'].to(config.device)
        edge_attr = batch['edge_attr'].to(config.device)
        batch_ptr = batch['batch_ptr'].to(config.device)
        
        _, pose_scores, _ = model(
            x, coords, accessibility, valid_mask,
            edge_index, edge_attr, batch_ptr
        )
        
        for b in range(pose_scores.shape[0]):
            mask = valid_mask[b].cpu().numpy()
            ps = pose_scores[b].cpu().numpy()[mask]
            labels = som_labels[b].numpy()[mask]
            all_pose_scores.extend(ps)
            all_som_labels.extend(labels)

all_pose_scores = np.array(all_pose_scores)
all_som_labels = np.array(all_som_labels)

# Plot distribution
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

som_scores = all_pose_scores[all_som_labels > 0]
non_som_scores = all_pose_scores[all_som_labels == 0]

ax.hist(non_som_scores, bins=50, alpha=0.5, label='Non-SoM atoms', density=True)
ax.hist(som_scores, bins=50, alpha=0.5, label='SoM atoms', density=True)
ax.set_xlabel('Reverse Geometry Score')
ax.set_ylabel('Density')
ax.set_title('What the Model Learned About Enzyme Pocket Dynamics')
ax.legend()
plt.tight_layout()
plt.savefig('reverse_geometry_analysis.png', dpi=150)
plt.show()

print(f"\nSoM atoms: mean={som_scores.mean():.3f}, std={som_scores.std():.3f}")
print(f"Non-SoM atoms: mean={non_som_scores.mean():.3f}, std={non_som_scores.std():.3f}")
print(f"\nSeparation: {som_scores.mean() - non_som_scores.mean():.3f}")